# Green alleys by distance to the nearest cat

This notebook follows Lab 5 (style it, layer it). It colours every **ruelle verte** by how far it is from the nearest cat I photographed. Dark green alleys have a cat close by, and pale ones are far from any cat.

```
data/ruelles_vertes.geojson        input: the alleys, from ruelles_pipeline.ipynb
data/cat_photos.geojson            input: the cats, from mvp_pipeline.ipynb
data/ruelles_cat_distance.geojson  output, written by this notebook
site/app.js                        colours each alley by dist_m
```

This uses Lab 5's **Recipe 8 (distance to the nearest something)** and adds **Recipe 9 (how many are within a distance)** for the popup. Run each cell in order, top to bottom. If you add new cats, run `mvp_pipeline.ipynb` first, then run this notebook again.

## 1) Import libraries

In [2]:
import os
import geopandas as gpd
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.2.0 is ready


## 2) Load both layers in metres

Distances must be measured in **metres**. In degrees a distance means nothing, so both layers go to Montréal's metre system (MTM zone 8, `EPSG:2950`) before anything is measured. As in Lab 5, metres are for measuring and degrees are for the web map at the end.

`cat_id` is the short ID that `app.js` and `gallery.js` already make from the photo's file name (`FullSizeRender 2.HEIC` becomes `FullSizeRender_2`). Keeping it lets an alley's popup link to its nearest cat.

In [3]:
alleys = gpd.read_file("../data/ruelles_vertes.geojson").to_crs("EPSG:2950")
cats = gpd.read_file("../data/cat_photos.geojson").to_crs("EPSG:2950")

cats["cat_id"] = (cats["photo"]
                  .str.replace(r"\.[^.]+$", "", regex=True)           # drop the extension
                  .str.replace(r"[^A-Za-z0-9_-]", "_", regex=True))   # same rule as catId() in app.js

print(len(alleys), "alley segments,", len(cats), "cats")
print("Both in metres:", alleys.crs.to_epsg(), cats.crs.to_epsg())

1262 alley segments, 73 cats
Both in metres: 2950 2950


## 3) Distance from each alley to the nearest cat (Recipe 8)

`sjoin_nearest` finds, for each alley, the closest cat and how far away it is. The distance is measured from the **edge** of the alley shape, so a cat photographed inside an alley gets 0 m.

This is straight-line distance ("as the crow flies"), not walking distance. The README says so too.

In [4]:
near = gpd.sjoin_nearest(alleys, cats[["cat_id", "geometry"]], distance_col="dist_m")
near = near[~near.index.duplicated()]                 # two cats exactly as close: keep one
near["dist_m"] = near["dist_m"].round(0).astype(int)
near = near.drop(columns="index_right").rename(columns={"cat_id": "nearest_cat"})

print("Alleys measured:", len(near), "of", len(alleys))   # must be the same

Alleys measured: 1262 of 1262


## 4) How many cats within 500 m of each alley (Recipe 9)

The nearest distance doesn't show whether there is one cat nearby or ten. So we also count every cat within **500 m**, about a 6-minute walk. We draw a 500 m buffer around each alley and count the cats that fall inside it with a spatial join, as in Lab 5's Layer A.

In [5]:
RADIUS = 500                                            # metres
zones = near[["alley_id", "geometry"]].copy()
zones["geometry"] = zones.buffer(RADIUS)                 # a 500 m band around each alley
zones["row"] = range(len(zones))                         # alley_id repeats (one alley, several segments), so count by row
j = gpd.sjoin(cats[["cat_id", "geometry"]], zones, predicate="intersects")
counts = j.groupby("row").size()

near["cats_500m"] = zones["row"].map(counts).fillna(0).astype(int).values
print("Alleys with at least one cat within 500 m:", (near["cats_500m"] > 0).sum())
print("Most cats near one alley:", near["cats_500m"].max())

Alleys with at least one cat within 500 m: 225
Most cats near one alley: 43


**Check:** an alley less than 500 m from its nearest cat must have at least one cat within 500 m, and an alley more than 500 m away must have none. (An alley exactly 500 m away could go either way after rounding, so it is left out.) If the two numbers below are not both 0, the two recipes disagree and something is wrong (usually a layer that was not converted to metres).

In [6]:
print("Close but counted 0:", ((near["dist_m"] < RADIUS) & (near["cats_500m"] == 0)).sum())
print("Far but counted some:", ((near["dist_m"] > RADIUS) & (near["cats_500m"] > 0)).sum())

Close but counted 0: 0
Far but counted some: 0


## 5) Look at the numbers before you choose colours

Colours on a map are classes: you decide where one class ends and the next begins. Read the spread first.

In [7]:
print(near["dist_m"].describe().round(0))
print()
for limit in [100, 250, 500, 1000]:
    print(f"Within {limit:>4} m of a cat:", (near["dist_m"] < limit).sum())
print("In the Plateau:", near["borough"].str.contains("Plateau").sum(), "segments, the farthest",
      near.loc[near["borough"].str.contains("Plateau"), "dist_m"].max(), "m from a cat")

count    1262.0
mean     1932.0
std      1654.0
min         0.0
25%       716.0
50%      1780.0
75%      2426.0
max      9227.0
Name: dist_m, dtype: float64

Within  100 m of a cat: 53
Within  250 m of a cat: 125
Within  500 m of a cat: 225
Within 1000 m of a cat: 408
In the Plateau: 281 segments, the farthest 1139 m from a cat


The cats cluster where I walk (Plateau and Mile End), so most of Montréal's alleys are more than 1 km from any cat. Almost all of the Plateau's alleys are within 1 km. The classes are in walking time, at about 80 m a minute, because the map is for planning a walk:

| Class | Distance to nearest cat | About |
| --- | --- | --- |
| 1 (darkest) | under 100 m | a cat right here |
| 2 | 100 to 249 m | under 3 minutes' walk |
| 3 | 250 to 499 m | under 6 minutes |
| 4 | 500 to 999 m | under 12 minutes |
| 5 (palest) | 1 km or more | far from any cat |

The page's colour function in `site/app.js` (`alleyColour`) uses exactly these breaks, and the map key in `index.html` must say the same thing.

## 6) Convert to longitude and latitude, then write

Back to degrees for the web, with only the fields the popup needs. **Output:** `../data/ruelles_cat_distance.geojson`

In [8]:
near["year"] = near["year"].astype("Int64")             # a whole number, so the popup says 2014, not 2014.0
out = near[["borough", "alley_id", "year", "circulation", "greening", "area_m2",
            "dist_m", "nearest_cat", "cats_500m", "geometry"]].to_crs("EPSG:4326")
out_path = "../data/ruelles_cat_distance.geojson"
out.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", out_path, "-", round(os.path.getsize(out_path) / 1_000_000, 2), "MB")

Wrote ../data/ruelles_cat_distance.geojson - 0.78 MB


## 7) Check the file

The check cell from Lab 4 and Lab 5, with the four settings changed for this file.

In [9]:
# ---- settings: change these for your own data ----
FILE = "../data/ruelles_cat_distance.geojson"
WEST, EAST, SOUTH, NORTH = -73.98, -73.47, 45.40, 45.71   # a box around the island of Montréal, in degrees
EXPECTED = 1262           # how many features you expect (same as ruelles_vertes.geojson)
POPUP_FIELDS = ["borough", "year", "dist_m", "nearest_cat", "cats_500m"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


## 8) Next: open the map

1. In VS Code, right-click `site/index.html` and choose **Open with Live Server**.
2. `site/app.js` now fetches `../data/ruelles_cat_distance.geojson` instead of `ruelles_vertes.geojson` and colours each alley with `alleyColour(dist_m)`. Click an alley to see how far its nearest cat is, then click **Meet this cat** to jump to it.